In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import mlflow
import mlflow.sklearn
from datetime import datetime

CATALOG = "intelligent_etl"
SCHEMA  = "default"
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S")

# MLflow experiment
mlflow.set_experiment("/intelligent_etl_anomaly_detection")
print(f"Run ID: {RUN_ID}")

2026/05/13 13:23:38 INFO mlflow.tracking.fluent: Experiment with name '/intelligent_etl_anomaly_detection' does not exist. Creating a new experiment.


Run ID: 20260513_132337


In [0]:
silver = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders")
print(f"Loaded {silver.count():,} rows from Silver")

# Collect to Pandas for sklearn
pdf = silver.toPandas()
print(f"Pandas shape: {pdf.shape}")

Loaded 9,879 rows from Silver
Pandas shape: (9879, 31)


In [0]:
DQ_COLS = [
    "dq_has_required_null",
    "dq_invalid_amount",
    "dq_future_date",
    "dq_invalid_status",
    "dq_invalid_region",
    "dq_invalid_quantity",
    "dq_invalid_discount",
    "dq_amount_mismatch",
]

pdf["rule_flag"] = pdf[DQ_COLS].any(axis=1).fillna(False)
print(f"Rule flag: {pdf['rule_flag'].sum():,} rows flagged")

Rule flag: 320 rows flagged


In [0]:
ML_FEATURES = ["quantity", "unit_price", "total_amount", "discount_pct", "_null_count"]

numeric = pdf[ML_FEATURES].fillna(0).astype(float)
means   = numeric.mean()
stds    = numeric.std().replace(0, 1)
zscores = (numeric - means) / stds

ZSCORE_THRESHOLD = 3.5
pdf["zscore_flag"]   = (zscores.abs() > ZSCORE_THRESHOLD).any(axis=1)
pdf["_max_zscore"]   = zscores.abs().max(axis=1).round(3)

print(f"Z-score flag: {pdf['zscore_flag'].sum():,} rows flagged")
print(f"\nTop 5 highest z-scores:")
print(pdf[["order_id","unit_price","total_amount","_max_zscore"]]
      .sort_values("_max_zscore", ascending=False).head())

Z-score flag: 143 rows flagged

Top 5 highest z-scores:
                                  order_id  ...  _max_zscore
2493  41ba8705-d28d-4a85-97de-58a80d7a9664  ...       51.107
9252  f0eb08fa-02ca-46db-ac49-004c29d23956  ...       42.280
1655  2c2d25be-4db9-4d78-81b0-eaf9d688dc12  ...       30.669
6556  aabfa144-820c-46a2-a348-6ddadc74264e  ...       30.279
6788  b14dcbf5-d637-41a7-a3bb-4af202bebe26  ...       26.049

[5 rows x 4 columns]


In [0]:
CONTAMINATION = 0.05

X        = numeric.values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

with mlflow.start_run(run_name=f"isolation_forest_{RUN_ID}"):
    model = IsolationForest(
        contamination=CONTAMINATION,
        n_estimators=200,
        random_state=42
    )
    model.fit(X_scaled)

    pdf["if_score"] = model.score_samples(X_scaled).round(4)
    pdf["if_flag"]  = model.predict(X_scaled) == -1

    # Log to MLflow
    mlflow.log_params({
        "contamination":    CONTAMINATION,
        "zscore_threshold": ZSCORE_THRESHOLD,
        "n_estimators":     200,
        "n_features":       len(ML_FEATURES),
        "features":         ",".join(ML_FEATURES),
    })
    mlflow.sklearn.log_model(model, "isolation_forest_model")

    print(f"Isolation Forest flag: {pdf['if_flag'].sum():,} rows flagged")

2026/05/13 13:24:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-b172f60e-80a2.cloud.databricks.com/ml/experiments/2048797709940972/models/m-e202ae69dc444252a646e9e3fbc17aaf?o=1477530146457703
2026/05/13 13:24:53 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


Isolation Forest flag: 494 rows flagged


In [0]:
from mlflow.models.signature import infer_signature

signature = infer_signature(X_scaled, model.predict(X_scaled))
input_example = X_scaled[:5]

mlflow.sklearn.log_model(
    model,
    "isolation_forest_model",
    signature=signature,
    input_example=input_example
)

2026/05/13 13:25:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-b172f60e-80a2.cloud.databricks.com/ml/experiments/2048797709940972/models/m-96f7151a748a409aadff4cee309e89ed?o=1477530146457703


In [0]:
# Final anomaly label — any signal fires
pdf["is_anomaly"] = pdf["rule_flag"] | pdf["zscore_flag"] | pdf["if_flag"]

# Build reason string
def build_reason(row):
    reasons = []
    if row["rule_flag"]:
        fired = [c for c in DQ_COLS if row.get(c)]
        reasons.append("RULE:" + "|".join(fired))
    if row["zscore_flag"]:
        reasons.append(f"ZSCORE:max={row['_max_zscore']}")
    if row["if_flag"]:
        reasons.append(f"IF:score={row['if_score']}")
    return ",".join(reasons)

pdf["anomaly_reason"] = pdf.apply(build_reason, axis=1)

# Summary
total       = len(pdf)
n_anomaly   = pdf["is_anomaly"].sum()
anomaly_rate = n_anomaly / total * 100

print(f"\n{'='*45}")
print(f"ANOMALY DETECTION SUMMARY")
print(f"{'='*45}")
print(f"Total records:     {total:,}")
print(f"Clean records:     {total - n_anomaly:,}")
print(f"Anomalies:         {n_anomaly:,}")
print(f"Anomaly rate:      {anomaly_rate:.1f}%")
print(f"\nBy signal:")
print(f"  Rule-based:      {pdf['rule_flag'].sum():,}")
print(f"  Z-score:         {pdf['zscore_flag'].sum():,}")
print(f"  Isolation Forest:{pdf['if_flag'].sum():,}")


ANOMALY DETECTION SUMMARY
Total records:     9,879
Clean records:     9,161
Anomalies:         718
Anomaly rate:      7.3%

By signal:
  Rule-based:      320
  Z-score:         143
  Isolation Forest:494


In [0]:
# Convert bool cols cleanly
for c in ["rule_flag","zscore_flag","if_flag","is_anomaly"]:
    pdf[c] = pdf[c].astype(bool)

# Back to Spark
scored_df = spark.createDataFrame(pdf)

# Write scored Silver table
(
    scored_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_orders_scored")
)

print(f"Scored table written: {spark.table(f'{CATALOG}.{SCHEMA}.silver_orders_scored').count():,} rows")

Scored table written: 9,879 rows
